# E7 — Cluster-Aware Statistical Inference
## Calib-BERT v2 Reconstruction

E7 is the formal inferential audit for the frozen E2–E6 evidence chain.

### Core principle

The Dreaddit official test set contains multiple rows from the same `post_id`.  
Therefore, treating all 715 rows as independent would understate uncertainty.

E7 uses **`post_id` as the resampling / randomization cluster**.

### Primary inferential endpoints

The confirmatory-style inferential audit is limited to the two probability-quality endpoints that were already designated primary earlier:

- **Brier score**
- **Negative log-likelihood (NLL)**

ECE remains descriptive because it is bin-dependent and especially unstable under small resamples.

### Statistical procedures

For paired condition comparisons on the same official-test instances:

1. **cluster bootstrap by `post_id`**, 10,000 replicates, for 95% percentile confidence intervals;
2. **paired cluster sign-flip randomization**, 10,000 draws, for two-sided p-values on per-example proper-score differences;
3. **Holm correction** within each predeclared inferential family.

### Inferential families

#### Family A — Cross-backbone calibration effects
For BERT, RoBERTa, and DeBERTa:
- Temperature Scaling vs Vanilla: Brier, NLL
- LS 0.10 vs Vanilla: Brier, NLL

12 multiplicity-controlled tests.

#### Family B — Domain-shift / OOD effects
Using pooled LODO predictions:
- OOD vs in-domain for Vanilla, Vanilla+TS, and LS0.10: Brier, NLL
- within OOD: TS vs Vanilla and LS0.10 vs Vanilla: Brier, NLL

10 multiplicity-controlled tests.

### Secondary uncertainty analyses

Cluster-bootstrap confidence intervals are also reported for:
- F1 deltas
- AURC deltas
- domain-macro AURC under LODO

These are **not added to the Holm families**.

### Important limitation

E7 quantifies **test-sample uncertainty conditional on the frozen trained models**.  
It does **not** estimate optimization / retraining-seed uncertainty. That limitation must remain explicit in the manuscript.

### E6 topic stability

E6 enters the integrated evidence table as a stability/limitation result.  
Because fine-grained semantic topic identities were resolution-sensitive, E7 does **not** perform post-hoc confirmatory p-values on individual topic hotspots.


In [ ]:
# ============================================================
# 0. USER CONFIGURATION
# ============================================================
from pathlib import Path

USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT_ROOT = Path("/content/drive/MyDrive/CalibBERT_E7")
    except Exception as e:
        print("Google Drive mount failed:", repr(e))
        PROJECT_ROOT = Path("/content/calibbert_v2_e7")
else:
    PROJECT_ROOT = Path("/content/calibbert_v2_e7")

INPUT_DIR = Path("/content")
WORK_DIR = PROJECT_ROOT / "work"
OUT_DIR = PROJECT_ROOT / "outputs_e7"

for p in [WORK_DIR, OUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

N_BOOT = 10_000
N_SIGNFLIP = 10_000
ALPHA = 0.05
RNG_SEED = 20260919

BACKBONES = ["BERT", "RoBERTa", "DeBERTa"]
E3_CONDITIONS = ["Vanilla", "Vanilla+TS", "LS0.10"]
E4_CONDITIONS = ["Vanilla", "Vanilla+TS", "LS0.10"]

print("Project root:", PROJECT_ROOT)
print("Outputs:", OUT_DIR)


## 1. Inputs

Upload these **three final frozen archives** to `/content/`:

1. final `E3_RESULTS.zip` — the repaired/frozen E3 package
2. `E4_RESULTS.zip`
3. `E6_RESULTS.zip`

If several E3 ZIPs are present, the notebook automatically prefers the archive containing:
- `E3_NUMERICAL_REPAIR_RECORD.json`
- tie-aware `e3_temperature_rank_invariance.csv`

No GPU is required for E7.


In [ ]:
import os
import sys
import gc
import json
import math
import random
import hashlib
import zipfile
import platform
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import f1_score

print("Python:", sys.version)
print("NumPy :", np.__version__)
print("pandas:", pd.__version__)


## 2. Find and validate final E3/E4/E6 archives

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def zip_names(path):
    with zipfile.ZipFile(path, "r") as z:
        return set(z.namelist())

def choose_e3_zip(directory):
    candidates = sorted(Path(directory).glob("E3_RESULTS*.zip"))
    if not candidates:
        raise FileNotFoundError("No E3_RESULTS*.zip found in /content.")

    scored = []
    for p in candidates:
        try:
            names = zip_names(p)
            score = 0
            if "E3_NUMERICAL_REPAIR_RECORD.json" in names:
                score += 10
            if "e3_temperature_rank_invariance.csv" in names:
                score += 3
            if "e3_prediction_level_long.csv" in names:
                score += 3
            if "e3_training_history.csv" in names:
                score += 1
            scored.append((score, p.stat().st_mtime, p))
        except Exception:
            pass

    if not scored:
        raise RuntimeError("E3 ZIP candidates could not be inspected.")

    scored.sort(reverse=True)
    return scored[0][2]

def choose_single_zip(directory, pattern):
    candidates = sorted(Path(directory).glob(pattern))
    if not candidates:
        raise FileNotFoundError(f"No archive matching {pattern} found in /content.")
    return max(candidates, key=lambda p: p.stat().st_mtime)

E3_ZIP = choose_e3_zip(INPUT_DIR)
E4_ZIP = choose_single_zip(INPUT_DIR, "E4_RESULTS*.zip")
E6_ZIP = choose_single_zip(INPUT_DIR, "E6_RESULTS*.zip")

print("E3:", E3_ZIP)
print("E4:", E4_ZIP)
print("E6:", E6_ZIP)

print("\nInput archive SHA-256")
for p in [E3_ZIP, E4_ZIP, E6_ZIP]:
    print(p.name, sha256_file(p))


## 3. Extract frozen archives and verify required evidence files

In [ ]:
E3_DIR = WORK_DIR / "E3"
E4_DIR = WORK_DIR / "E4"
E6_DIR = WORK_DIR / "E6"

for d in [E3_DIR, E4_DIR, E6_DIR]:
    d.mkdir(parents=True, exist_ok=True)

for archive, dest in [
    (E3_ZIP, E3_DIR),
    (E4_ZIP, E4_DIR),
    (E6_ZIP, E6_DIR),
]:
    with zipfile.ZipFile(archive, "r") as z:
        z.extractall(dest)

required = {
    "E3": [
        E3_DIR / "e3_prediction_level_long.csv",
        E3_DIR / "e3_official_test_metrics.csv",
        E3_DIR / "e3_temperature_rank_invariance.csv",
    ],
    "E4": [
        E4_DIR / "e4_pooled_lodo_target_predictions_for_E7.csv",
        E4_DIR / "e4_prediction_level_long.csv",
        E4_DIR / "e4_lodo_metrics_long.csv",
        E4_DIR / "e4_macro_domain_summary.csv",
    ],
    "E6": [
        E6_DIR / "e6_top2vec_stability_summary.csv",
        E6_DIR / "e6_bertopic_stability_summary.csv",
        E6_DIR / "e6_top2vec_run_summary.csv",
        E6_DIR / "Table_E6_Aligned_Local_Reliability_Stability.csv",
    ],
}

for stage, files in required.items():
    for p in files:
        if not p.exists():
            raise FileNotFoundError(f"{stage}: required file missing: {p.name}")

print("Required frozen evidence files: PASS")


## 4. Load and align prediction-level evidence

The same 715 official-test instances must align by:
- `id`
- `post_id`
- `true_label`

The cluster unit is `post_id`.


In [ ]:
e3 = pd.read_csv(E3_DIR / "e3_prediction_level_long.csv")
e4_target = pd.read_csv(
    E4_DIR / "e4_pooled_lodo_target_predictions_for_E7.csv"
)
e4_long = pd.read_csv(E4_DIR / "e4_prediction_level_long.csv")

EXPECTED_N = 715

# E3 alignment
e3_counts = (
    e3.groupby(["backbone", "condition"])["id"].nunique()
)
if not e3_counts.eq(EXPECTED_N).all():
    raise RuntimeError(f"E3 does not have {EXPECTED_N} unique IDs per condition.")

# E4 pooled alignment
e4_counts = e4_target.groupby("condition")["id"].nunique()
if not e4_counts.eq(EXPECTED_N).all():
    raise RuntimeError(f"E4 pooled target does not have {EXPECTED_N} unique IDs per condition.")

# Anchor identity from BERT Vanilla
anchor = (
    e3[
        (e3["backbone"] == "BERT")
        & (e3["condition"] == "Vanilla")
    ][["id", "post_id", "true_label"]]
    .sort_values("id")
    .reset_index(drop=True)
)

for condition in E4_CONDITIONS:
    x = (
        e4_target[e4_target["condition"] == condition]
        [["id", "post_id", "true_label"]]
        .sort_values("id")
        .reset_index(drop=True)
    )

    if not anchor.equals(x):
        raise RuntimeError(
            f"E3/E4 identity mismatch for E4 condition {condition}."
        )

N_CLUSTERS = int(anchor["post_id"].nunique())

print("Official-test rows:", EXPECTED_N)
print("Unique post_id clusters:", N_CLUSTERS)
print("Repeated rows beyond unique clusters:", EXPECTED_N - N_CLUSTERS)
print("E3/E4 paired identity alignment: PASS")


## 5. Write E7 analysis lock before inference

In [ ]:
e7_lock = {
    "analysis": "E7 Cluster-Aware Statistical Inference",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "inputs": {
        "E3_archive": {
            "filename": E3_ZIP.name,
            "sha256": sha256_file(E3_ZIP),
        },
        "E4_archive": {
            "filename": E4_ZIP.name,
            "sha256": sha256_file(E4_ZIP),
        },
        "E6_archive": {
            "filename": E6_ZIP.name,
            "sha256": sha256_file(E6_ZIP),
        },
    },
    "n_test_rows": EXPECTED_N,
    "cluster_unit": "post_id",
    "n_unique_clusters": N_CLUSTERS,
    "bootstrap_replicates": N_BOOT,
    "cluster_signflip_draws": N_SIGNFLIP,
    "rng_seed": RNG_SEED,
    "confidence_level": 0.95,
    "primary_inferential_endpoints": ["brier", "nll"],
    "ece_role": "descriptive only; bin-dependent",
    "family_A": {
        "name": "cross_backbone_calibration",
        "tests": (
            "For each of BERT/RoBERTa/DeBERTa: "
            "(Vanilla+TS minus Vanilla) and (LS0.10 minus Vanilla), "
            "for Brier and NLL"
        ),
        "n_tests": 12,
        "multiplicity": "Holm within family",
    },
    "family_B": {
        "name": "domain_shift_calibration",
        "tests": (
            "OOD minus in-domain for each condition (3 x Brier/NLL), "
            "plus within-OOD TS-minus-Vanilla and LS-minus-Vanilla "
            "(2 x Brier/NLL)"
        ),
        "n_tests": 10,
        "multiplicity": "Holm within family",
    },
    "secondary_endpoints": [
        "F1 delta with cluster-bootstrap CI",
        "AURC delta with cluster-bootstrap CI",
        "domain-macro AURC delta with domain-stratified cluster-bootstrap CI",
    ],
    "topic_level_confirmatory_testing": False,
    "topic_reason": (
        "E6 demonstrated resolution-sensitive fine-grained topic identities; "
        "topic evidence remains a stability/diagnostic result."
    ),
    "uncertainty_scope": (
        "Conditional on frozen trained models; captures clustered official-test "
        "sampling uncertainty, not retraining/optimization-seed uncertainty."
    ),
}

(OUT_DIR / "E7_ANALYSIS_LOCK.json").write_text(
    json.dumps(e7_lock, indent=2)
)

print(json.dumps(e7_lock, indent=2))


## 6. Statistical helper functions

In [ ]:
EPS = 1e-12

def clipped_p(p):
    return np.clip(np.asarray(p, dtype=float), EPS, 1 - EPS)

def obs_brier(y, p):
    p = clipped_p(p)
    return (p - y) ** 2

def obs_nll(y, p):
    p = clipped_p(p)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

def aurc_metric(y, pred, confidence):
    y = np.asarray(y, dtype=int)
    pred = np.asarray(pred, dtype=int)
    confidence = np.asarray(confidence, dtype=float)

    order = np.lexsort((np.arange(len(y)), -confidence))
    errors = (pred[order] != y[order]).astype(float)
    cumulative_risk = np.cumsum(errors) / np.arange(1, len(y) + 1)
    return float(np.mean(cumulative_risk))

def f1_metric(y, pred):
    return float(f1_score(y, pred, zero_division=0))

def stable_seed(*parts, offset=0):
    payload = "||".join(map(str, parts)).encode("utf-8")
    digest = hashlib.sha256(payload).digest()
    value = int.from_bytes(digest[:8], "big")
    return int((RNG_SEED + offset + value) % (2**32 - 1))

def aligned_pair(df_a, df_b):
    key = ["id", "post_id", "true_label"]

    a = df_a.sort_values("id").reset_index(drop=True)
    b = df_b.sort_values("id").reset_index(drop=True)

    if not a[key].equals(b[key]):
        raise RuntimeError("Paired frames are not aligned on id/post_id/true_label.")

    return a, b

def cluster_structure(post_ids):
    post_ids = np.asarray(post_ids)
    unique_clusters = pd.unique(post_ids)
    cluster_indices = {
        c: np.flatnonzero(post_ids == c)
        for c in unique_clusters
    }
    return unique_clusters, cluster_indices

def cluster_bootstrap_mean_delta(
    post_ids,
    per_obs_delta,
    n_boot=N_BOOT,
    seed=RNG_SEED,
):
    """
    Row-weighted mean delta with a nonparametric cluster bootstrap.
    Whole post_id clusters are resampled with replacement.
    """
    post_ids = np.asarray(post_ids)
    d = np.asarray(per_obs_delta, dtype=float)

    clusters = pd.unique(post_ids)
    cluster_sums = np.array(
        [d[post_ids == c].sum() for c in clusters],
        dtype=float,
    )
    cluster_sizes = np.array(
        [(post_ids == c).sum() for c in clusters],
        dtype=float,
    )

    rng = np.random.default_rng(seed)
    reps = np.empty(n_boot, dtype=float)

    chunk = 1000
    for start in range(0, n_boot, chunk):
        n = min(chunk, n_boot - start)
        sampled_idx = rng.integers(
            0,
            len(clusters),
            size=(n, len(clusters)),
        )

        numerator = cluster_sums[sampled_idx].sum(axis=1)
        denominator = cluster_sizes[sampled_idx].sum(axis=1)
        reps[start:start+n] = numerator / denominator

    point = float(np.mean(d))
    lo, hi = np.quantile(reps, [0.025, 0.975])

    return point, float(lo), float(hi), reps

def cluster_signflip_pvalue(
    post_ids,
    per_obs_delta,
    n_draws=N_SIGNFLIP,
    seed=RNG_SEED + 1,
):
    post_ids = np.asarray(post_ids)
    d = np.asarray(per_obs_delta, dtype=float)

    cluster_ids = pd.unique(post_ids)
    cluster_sums = np.array(
        [d[post_ids == c].sum() for c in cluster_ids],
        dtype=float,
    )

    observed = abs(float(d.mean()))
    denominator = len(d)

    rng = np.random.default_rng(seed)
    extreme = 0

    chunk = 1000
    for start in range(0, n_draws, chunk):
        n = min(chunk, n_draws - start)
        signs = rng.choice(
            np.array([-1.0, 1.0]),
            size=(n, len(cluster_ids)),
            replace=True,
        )

        permuted = np.abs(
            (signs @ cluster_sums) / denominator
        )

        extreme += int(np.sum(permuted >= observed - 1e-15))

    return float((extreme + 1) / (n_draws + 1))

def bootstrap_metric_delta(
    df_a,
    df_b,
    metric,
    n_boot=N_BOOT,
    seed=RNG_SEED + 10,
):
    a, b = aligned_pair(df_a, df_b)

    post_ids = a["post_id"].to_numpy()
    clusters, idx_map = cluster_structure(post_ids)

    y = a["true_label"].to_numpy(int)

    def evaluate(frame, idx):
        yy = y[idx]

        if metric == "f1":
            return f1_metric(
                yy,
                frame["predicted_label"].to_numpy(int)[idx],
            )

        if metric == "aurc":
            return aurc_metric(
                yy,
                frame["predicted_label"].to_numpy(int)[idx],
                frame["confidence"].to_numpy(float)[idx],
            )

        raise ValueError(metric)

    full_idx = np.arange(len(a))
    point = evaluate(b, full_idx) - evaluate(a, full_idx)

    rng = np.random.default_rng(seed)
    reps = np.empty(n_boot, dtype=float)

    for r in range(n_boot):
        sampled = rng.choice(
            clusters,
            size=len(clusters),
            replace=True,
        )
        idx = np.concatenate([idx_map[c] for c in sampled])
        reps[r] = evaluate(b, idx) - evaluate(a, idx)

    lo, hi = np.quantile(reps, [0.025, 0.975])

    return float(point), float(lo), float(hi), reps

def holm_adjust(pvalues):
    p = np.asarray(pvalues, dtype=float)
    m = len(p)

    order = np.argsort(p)
    adjusted = np.empty(m, dtype=float)

    running = 0.0
    for rank, idx in enumerate(order):
        value = (m - rank) * p[idx]
        running = max(running, value)
        adjusted[idx] = min(1.0, running)

    return adjusted

def ci_excludes_zero(lo, hi):
    return bool((lo > 0) or (hi < 0))


## 7. Build aligned E3 condition frames

In [ ]:
def get_e3(backbone, condition):
    x = e3[
        (e3["backbone"] == backbone)
        & (e3["condition"] == condition)
    ].copy()

    if len(x) != EXPECTED_N:
        raise RuntimeError(
            f"E3 {backbone}/{condition}: expected {EXPECTED_N}, got {len(x)}"
        )

    return x.sort_values("id").reset_index(drop=True)

e3_frames = {
    (b, c): get_e3(b, c)
    for b in BACKBONES
    for c in E3_CONDITIONS
}

print("E3 aligned condition frames:", len(e3_frames))


## 8. Family A — cross-backbone proper-score inference

Effect definition:

\[
\Delta = 	ext{intervention} - 	ext{Vanilla}
\]

For Brier/NLL:
- **negative Δ** = lower proper loss under the intervention
- **positive Δ** = higher proper loss under the intervention


In [ ]:
family_a_rows = []

comparison_specs = [
    ("Vanilla+TS", "TS_minus_Vanilla"),
    ("LS0.10", "LS_minus_Vanilla"),
]

for backbone in BACKBONES:
    base = e3_frames[(backbone, "Vanilla")]

    y = base["true_label"].to_numpy(int)
    post_ids = base["post_id"].to_numpy()

    for condition, comparison in comparison_specs:
        comp = e3_frames[(backbone, condition)]
        base, comp = aligned_pair(base, comp)

        for metric in ["brier", "nll"]:
            if metric == "brier":
                loss_base = obs_brier(
                    y,
                    base["p_stress"].to_numpy(float),
                )
                loss_comp = obs_brier(
                    y,
                    comp["p_stress"].to_numpy(float),
                )
            else:
                loss_base = obs_nll(
                    y,
                    base["p_stress"].to_numpy(float),
                )
                loss_comp = obs_nll(
                    y,
                    comp["p_stress"].to_numpy(float),
                )

            delta_obs = loss_comp - loss_base

            point, lo, hi, _ = cluster_bootstrap_mean_delta(
                post_ids,
                delta_obs,
                seed=stable_seed(backbone, condition, metric, "familyA_boot"),
            )

            p = cluster_signflip_pvalue(
                post_ids,
                delta_obs,
                seed=stable_seed(backbone, condition, metric, "familyA_sign", offset=100000),
            )

            family_a_rows.append({
                "family": "A_cross_backbone_calibration",
                "backbone": backbone,
                "comparison": comparison,
                "metric": metric,
                "effect_definition": "intervention_minus_vanilla",
                "delta": point,
                "ci95_low": lo,
                "ci95_high": hi,
                "cluster_signflip_p": p,
                "ci_excludes_zero": ci_excludes_zero(lo, hi),
                "n_rows": EXPECTED_N,
                "n_post_id_clusters": N_CLUSTERS,
            })

family_a = pd.DataFrame(family_a_rows)
family_a["holm_p"] = holm_adjust(
    family_a["cluster_signflip_p"].to_numpy()
)
family_a["holm_reject_0.05"] = (
    family_a["holm_p"] < ALPHA
)

family_a.to_csv(
    OUT_DIR / "Table_E7_FamilyA_CrossBackbone_ProperScores.csv",
    index=False,
)

display(family_a)


## 9. Build aligned E4 pooled OOD and BERT in-domain frames

In [ ]:
def get_e4_target(condition):
    x = e4_target[
        e4_target["condition"] == condition
    ].copy()

    if len(x) != EXPECTED_N:
        raise RuntimeError(
            f"E4 pooled {condition}: expected {EXPECTED_N}, got {len(x)}"
        )

    return x.sort_values("id").reset_index(drop=True)

e4_frames = {
    c: get_e4_target(c)
    for c in E4_CONDITIONS
}

id_bert_frames = {
    c: e3_frames[("BERT", c)].copy()
    for c in E3_CONDITIONS
}

condition_pairs = {
    "Vanilla": "Vanilla",
    "Vanilla+TS": "Vanilla+TS",
    "LS0.10": "LS0.10",
}

for e4_c, e3_c in condition_pairs.items():
    aligned_pair(
        id_bert_frames[e3_c],
        e4_frames[e4_c],
    )

print("E4 pooled OOD and E3/BERT in-domain frames aligned: PASS")


## 10. Family B — domain-shift / OOD proper-score inference

Two question types are tested:

### B1. OOD penalty
\[
\Delta = 	ext{LODO OOD} - 	ext{in-domain}
\]

Positive Brier/NLL delta indicates worse probability quality under domain shift.

### B2. Calibration intervention within OOD
\[
\Delta = 	ext{intervention}_{OOD} - 	ext{Vanilla}_{OOD}
\]

Negative delta indicates lower proper loss under OOD.


In [ ]:
family_b_rows = []

# B1 — OOD minus in-domain for all three conditions
for condition in E4_CONDITIONS:
    id_frame = id_bert_frames[condition]
    ood_frame = e4_frames[condition]
    id_frame, ood_frame = aligned_pair(id_frame, ood_frame)

    y = id_frame["true_label"].to_numpy(int)
    post_ids = id_frame["post_id"].to_numpy()

    for metric in ["brier", "nll"]:
        if metric == "brier":
            id_loss = obs_brier(
                y,
                id_frame["p_stress"].to_numpy(float),
            )
            ood_loss = obs_brier(
                y,
                ood_frame["p_stress"].to_numpy(float),
            )
        else:
            id_loss = obs_nll(
                y,
                id_frame["p_stress"].to_numpy(float),
            )
            ood_loss = obs_nll(
                y,
                ood_frame["p_stress"].to_numpy(float),
            )

        delta_obs = ood_loss - id_loss

        point, lo, hi, _ = cluster_bootstrap_mean_delta(
            post_ids,
            delta_obs,
            seed=stable_seed(condition, metric, "ood_penalty_boot", offset=200000),
        )

        p = cluster_signflip_pvalue(
            post_ids,
            delta_obs,
            seed=stable_seed(condition, metric, "ood_penalty_sign", offset=300000),
        )

        family_b_rows.append({
            "family": "B_domain_shift_calibration",
            "question": "OOD_minus_in_domain",
            "condition": condition,
            "comparison": f"{condition}_OOD_minus_{condition}_ID",
            "metric": metric,
            "effect_definition": "ood_minus_in_domain",
            "delta": point,
            "ci95_low": lo,
            "ci95_high": hi,
            "cluster_signflip_p": p,
            "ci_excludes_zero": ci_excludes_zero(lo, hi),
            "n_rows": EXPECTED_N,
            "n_post_id_clusters": N_CLUSTERS,
        })

# B2 — intervention minus Vanilla within pooled OOD
ood_base = e4_frames["Vanilla"]

for condition, label in [
    ("Vanilla+TS", "TS_minus_Vanilla_OOD"),
    ("LS0.10", "LS_minus_Vanilla_OOD"),
]:
    comp = e4_frames[condition]
    ood_base, comp = aligned_pair(ood_base, comp)

    y = ood_base["true_label"].to_numpy(int)
    post_ids = ood_base["post_id"].to_numpy()

    for metric in ["brier", "nll"]:
        if metric == "brier":
            base_loss = obs_brier(
                y,
                ood_base["p_stress"].to_numpy(float),
            )
            comp_loss = obs_brier(
                y,
                comp["p_stress"].to_numpy(float),
            )
        else:
            base_loss = obs_nll(
                y,
                ood_base["p_stress"].to_numpy(float),
            )
            comp_loss = obs_nll(
                y,
                comp["p_stress"].to_numpy(float),
            )

        delta_obs = comp_loss - base_loss

        point, lo, hi, _ = cluster_bootstrap_mean_delta(
            post_ids,
            delta_obs,
            seed=stable_seed(condition, metric, "ood_intervention_boot", offset=400000),
        )

        p = cluster_signflip_pvalue(
            post_ids,
            delta_obs,
            seed=stable_seed(condition, metric, "ood_intervention_sign", offset=500000),
        )

        family_b_rows.append({
            "family": "B_domain_shift_calibration",
            "question": "intervention_within_OOD",
            "condition": condition,
            "comparison": label,
            "metric": metric,
            "effect_definition": "intervention_minus_vanilla_ood",
            "delta": point,
            "ci95_low": lo,
            "ci95_high": hi,
            "cluster_signflip_p": p,
            "ci_excludes_zero": ci_excludes_zero(lo, hi),
            "n_rows": EXPECTED_N,
            "n_post_id_clusters": N_CLUSTERS,
        })

family_b = pd.DataFrame(family_b_rows)
family_b["holm_p"] = holm_adjust(
    family_b["cluster_signflip_p"].to_numpy()
)
family_b["holm_reject_0.05"] = (
    family_b["holm_p"] < ALPHA
)

family_b.to_csv(
    OUT_DIR / "Table_E7_FamilyB_DomainShift_ProperScores.csv",
    index=False,
)

display(family_b)


## 11. Secondary E3 cluster-bootstrap inference: F1 and AURC

Temperature Scaling is expected to preserve class decisions and within-model confidence ranking.

Therefore:
- TS vs Vanilla F1/AURC is recorded structurally;
- LS vs Vanilla receives cluster-bootstrap CIs.


In [ ]:
secondary_e3_rows = []

for backbone in BACKBONES:
    vanilla = e3_frames[(backbone, "Vanilla")]
    ts = e3_frames[(backbone, "Vanilla+TS")]
    ls = e3_frames[(backbone, "LS0.10")]

    vanilla, ts = aligned_pair(vanilla, ts)
    vanilla, ls = aligned_pair(vanilla, ls)

    # Structural TS differences
    for metric in ["f1", "aurc"]:
        if metric == "f1":
            point = (
                f1_metric(
                    ts["true_label"],
                    ts["predicted_label"],
                )
                - f1_metric(
                    vanilla["true_label"],
                    vanilla["predicted_label"],
                )
            )
        else:
            point = (
                aurc_metric(
                    ts["true_label"],
                    ts["predicted_label"],
                    ts["confidence"],
                )
                - aurc_metric(
                    vanilla["true_label"],
                    vanilla["predicted_label"],
                    vanilla["confidence"],
                )
            )

        secondary_e3_rows.append({
            "backbone": backbone,
            "comparison": "TS_minus_Vanilla",
            "metric": metric,
            "delta": float(point),
            "ci95_low": float(point),
            "ci95_high": float(point),
            "inference_note": "structural equality / invariance audit",
        })

    # LS bootstrap
    for metric in ["f1", "aurc"]:
        point, lo, hi, _ = bootstrap_metric_delta(
            vanilla,
            ls,
            metric=metric,
            seed=stable_seed(backbone, metric, "secondary_e3", offset=600000),
        )

        secondary_e3_rows.append({
            "backbone": backbone,
            "comparison": "LS_minus_Vanilla",
            "metric": metric,
            "delta": point,
            "ci95_low": lo,
            "ci95_high": hi,
            "inference_note": "post_id cluster bootstrap; no Holm test",
        })

secondary_e3 = pd.DataFrame(secondary_e3_rows)

secondary_e3.to_csv(
    OUT_DIR / "Table_E7_Secondary_E3_F1_AURC.csv",
    index=False,
)

display(secondary_e3)


## 12. Domain-stratified cluster bootstrap for macro-domain AURC

For E4, pooling fold-specific Temperature-Scaled probabilities can re-rank examples across folds because each held-out domain has its own temperature.

Therefore E7 evaluates selective reliability under shift with **equal-weight macro-domain AURC**:

1. within each of the five held-out domains, resample `post_id` clusters with replacement;
2. compute AURC inside that domain;
3. average the five domain AURCs equally.

This preserves the intended LODO structure.


In [ ]:
# Domain labels are available in E4 pooled target predictions.
domains = sorted(e4_target["domain"].unique().tolist())

def macro_domain_aurc(frame):
    vals = []
    for domain in domains:
        x = frame[frame["domain"] == domain]
        vals.append(
            aurc_metric(
                x["true_label"].to_numpy(int),
                x["predicted_label"].to_numpy(int),
                x["confidence"].to_numpy(float),
            )
        )
    return float(np.mean(vals))

def stratified_domain_cluster_bootstrap_delta(
    frame_a,
    frame_b,
    n_boot=N_BOOT,
    seed=RNG_SEED + 700000,
):
    a, b = aligned_pair(frame_a, frame_b)

    # Add domain from E4 target frame if missing.
    if "domain" not in a.columns:
        domain_map = (
            e4_target[["id", "domain"]]
            .drop_duplicates("id")
        )
        a = a.merge(domain_map, on="id", how="left")
    if "domain" not in b.columns:
        domain_map = (
            e4_target[["id", "domain"]]
            .drop_duplicates("id")
        )
        b = b.merge(domain_map, on="id", how="left")

    point = macro_domain_aurc(b) - macro_domain_aurc(a)

    rng = np.random.default_rng(seed)
    reps = np.empty(n_boot, dtype=float)

    # Precompute row indices by domain and post cluster.
    structures = {}

    for domain in domains:
        idx_domain = np.flatnonzero(a["domain"].to_numpy() == domain)
        post_ids = a.iloc[idx_domain]["post_id"].to_numpy()

        clusters = pd.unique(post_ids)
        idx_map_local = {
            c: idx_domain[np.flatnonzero(post_ids == c)]
            for c in clusters
        }

        structures[domain] = (clusters, idx_map_local)

    for r in range(n_boot):
        domain_deltas = []

        for domain in domains:
            clusters, idx_map = structures[domain]

            sampled = rng.choice(
                clusters,
                size=len(clusters),
                replace=True,
            )

            idx = np.concatenate(
                [idx_map[c] for c in sampled]
            )

            aurc_a = aurc_metric(
                a["true_label"].to_numpy(int)[idx],
                a["predicted_label"].to_numpy(int)[idx],
                a["confidence"].to_numpy(float)[idx],
            )

            aurc_b = aurc_metric(
                b["true_label"].to_numpy(int)[idx],
                b["predicted_label"].to_numpy(int)[idx],
                b["confidence"].to_numpy(float)[idx],
            )

            domain_deltas.append(aurc_b - aurc_a)

        reps[r] = float(np.mean(domain_deltas))

    lo, hi = np.quantile(reps, [0.025, 0.975])

    return float(point), float(lo), float(hi), reps

# Add domain to BERT in-domain frames.
domain_map = (
    e4_target[["id", "domain"]]
    .drop_duplicates("id")
)

id_with_domain = {
    condition: (
        id_bert_frames[condition]
        .merge(domain_map, on="id", how="left")
    )
    for condition in E3_CONDITIONS
}

ood_macro_rows = []

# OOD penalty for Vanilla macro AURC
point, lo, hi, _ = stratified_domain_cluster_bootstrap_delta(
    id_with_domain["Vanilla"],
    e4_frames["Vanilla"],
    seed=RNG_SEED + 710001,
)

ood_macro_rows.append({
    "comparison": "Vanilla_OOD_minus_Vanilla_ID",
    "metric": "macro_domain_aurc",
    "delta": point,
    "ci95_low": lo,
    "ci95_high": hi,
})

# Interventions within OOD
for condition, label, seed_offset in [
    ("Vanilla+TS", "TS_minus_Vanilla_OOD", 710002),
    ("LS0.10", "LS_minus_Vanilla_OOD", 710003),
]:
    point, lo, hi, _ = stratified_domain_cluster_bootstrap_delta(
        e4_frames["Vanilla"],
        e4_frames[condition],
        seed=RNG_SEED + seed_offset,
    )

    ood_macro_rows.append({
        "comparison": label,
        "metric": "macro_domain_aurc",
        "delta": point,
        "ci95_low": lo,
        "ci95_high": hi,
    })

ood_macro_aurc = pd.DataFrame(ood_macro_rows)

ood_macro_aurc.to_csv(
    OUT_DIR / "Table_E7_Secondary_E4_MacroDomain_AURC.csv",
    index=False,
)

display(ood_macro_aurc)


## 13. E6 stability evidence — integrated without topic-level p-hacking

E6 is summarized as:
- Top2Vec seed stability
- BERTopic seed stability
- Top2Vec topic-resolution distribution
- recovery of seed-42 aligned topics

No individual semantic hotspot receives a confirmatory p-value.


In [ ]:
top_stability = pd.read_csv(
    E6_DIR / "e6_top2vec_stability_summary.csv"
)
ber_stability = pd.read_csv(
    E6_DIR / "e6_bertopic_stability_summary.csv"
)
top_runs = pd.read_csv(
    E6_DIR / "e6_top2vec_run_summary.csv"
)
aligned_local = pd.read_csv(
    E6_DIR / "Table_E6_Aligned_Local_Reliability_Stability.csv"
)

topic_count_distribution = (
    top_runs["n_topics_development"]
    .value_counts()
    .sort_index()
    .rename_axis("n_topics")
    .reset_index(name="n_seeds")
)
topic_count_distribution["fraction"] = (
    topic_count_distribution["n_seeds"]
    / len(top_runs)
)

robust_aligned = aligned_local[
    aligned_local["seed_recovery_fraction"] >= 0.70
].copy()

e6_summary_rows = []

for _, r in top_stability.iterrows():
    e6_summary_rows.append({
        "stage": "E6",
        "evidence_type": "Top2Vec seed stability",
        "scope": r["corpus"],
        "estimate_1_name": "ARI mean",
        "estimate_1": r["ari_mean"],
        "estimate_2_name": "NMI mean",
        "estimate_2": r["nmi_mean"],
        "note": "descriptive partition stability",
    })

for _, r in ber_stability.iterrows():
    e6_summary_rows.append({
        "stage": "E6",
        "evidence_type": "BERTopic seed stability",
        "scope": r["corpus"],
        "estimate_1_name": "ARI mean",
        "estimate_1": r["ari_mean"],
        "estimate_2_name": "NMI mean",
        "estimate_2": r["nmi_mean"],
        "note": "descriptive alternative-method stability",
    })

e6_integrated = pd.DataFrame(e6_summary_rows)

e6_integrated.to_csv(
    OUT_DIR / "Table_E7_E6_Stability_Evidence.csv",
    index=False,
)
topic_count_distribution.to_csv(
    OUT_DIR / "Table_E7_E6_Top2Vec_TopicCount_Distribution.csv",
    index=False,
)
robust_aligned.to_csv(
    OUT_DIR / "Table_E7_E6_RobustAlignedTopics_Descriptive.csv",
    index=False,
)

display(topic_count_distribution)
display(e6_integrated)


## 14. Integrated E2–E6 evidence table

This table consolidates inferential and stability evidence without collapsing everything into a single score.


In [ ]:
integrated_rows = []

# Family A
for _, r in family_a.iterrows():
    integrated_rows.append({
        "stage": "E2/E3",
        "evidence": (
            f"{r['backbone']} {r['comparison']} — {r['metric']}"
        ),
        "effect_definition": r["effect_definition"],
        "estimate": r["delta"],
        "ci95_low": r["ci95_low"],
        "ci95_high": r["ci95_high"],
        "holm_p": r["holm_p"],
        "holm_reject_0.05": r["holm_reject_0.05"],
        "inference_type": "cluster-aware proper-score inference",
        "scope_note": "conditional on frozen trained model",
    })

# Family B
for _, r in family_b.iterrows():
    integrated_rows.append({
        "stage": "E4",
        "evidence": (
            f"{r['comparison']} — {r['metric']}"
        ),
        "effect_definition": r["effect_definition"],
        "estimate": r["delta"],
        "ci95_low": r["ci95_low"],
        "ci95_high": r["ci95_high"],
        "holm_p": r["holm_p"],
        "holm_reject_0.05": r["holm_reject_0.05"],
        "inference_type": "cluster-aware proper-score inference",
        "scope_note": "LODO OOD; conditional on frozen fold models",
    })

# E5/E3 selective
for _, r in secondary_e3.iterrows():
    integrated_rows.append({
        "stage": "E5/E3",
        "evidence": (
            f"{r['backbone']} {r['comparison']} — {r['metric']}"
        ),
        "effect_definition": "intervention_minus_vanilla",
        "estimate": r["delta"],
        "ci95_low": r["ci95_low"],
        "ci95_high": r["ci95_high"],
        "holm_p": np.nan,
        "holm_reject_0.05": np.nan,
        "inference_type": r["inference_note"],
        "scope_note": "secondary selective/discrimination endpoint",
    })

# E4 macro AURC
for _, r in ood_macro_aurc.iterrows():
    integrated_rows.append({
        "stage": "E4",
        "evidence": (
            f"{r['comparison']} — macro-domain AURC"
        ),
        "effect_definition": "second_condition_minus_first",
        "estimate": r["delta"],
        "ci95_low": r["ci95_low"],
        "ci95_high": r["ci95_high"],
        "holm_p": np.nan,
        "holm_reject_0.05": np.nan,
        "inference_type": "domain-stratified post_id cluster bootstrap",
        "scope_note": "secondary selective endpoint",
    })

integrated = pd.DataFrame(integrated_rows)

integrated.to_csv(
    OUT_DIR / "Table_E7_Integrated_Evidence_E2_to_E6.csv",
    index=False,
)

display(integrated)


## 15. Figures — multiplicity-controlled proper-score effects

Negative values for intervention-minus-Vanilla Brier/NLL indicate lower proper loss.

For OOD-minus-in-domain comparisons, positive values indicate an OOD penalty.


In [ ]:
# Family A figure
fig, ax = plt.subplots(figsize=(10, 6))

fa = family_a.copy()
fa["label"] = (
    fa["backbone"]
    + " | "
    + fa["comparison"]
    + " | "
    + fa["metric"]
)

x = np.arange(len(fa))

ax.errorbar(
    x,
    fa["delta"],
    yerr=[
        np.maximum(fa["delta"] - fa["ci95_low"], 0),
        np.maximum(fa["ci95_high"] - fa["delta"], 0),
    ],
    fmt="o",
    capsize=3,
)

ax.axhline(0, linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(fa["label"], rotation=75, ha="right")
ax.set_ylabel("Paired proper-score delta")
ax.set_title("E7 Family A — Cross-Backbone Calibration Effects")
ax.grid(True, alpha=0.25)
fig.tight_layout()

fig.savefig(
    OUT_DIR / "Figure_E7_FamilyA_ProperScore_Effects.png",
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    OUT_DIR / "Figure_E7_FamilyA_ProperScore_Effects.pdf",
    bbox_inches="tight",
)
plt.show()

# Family B figure
fig, ax = plt.subplots(figsize=(10, 6))

fb = family_b.copy()
fb["label"] = (
    fb["comparison"]
    + " | "
    + fb["metric"]
)

x = np.arange(len(fb))

ax.errorbar(
    x,
    fb["delta"],
    yerr=[
        np.maximum(fb["delta"] - fb["ci95_low"], 0),
        np.maximum(fb["ci95_high"] - fb["delta"], 0),
    ],
    fmt="o",
    capsize=3,
)

ax.axhline(0, linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(fb["label"], rotation=75, ha="right")
ax.set_ylabel("Paired proper-score delta")
ax.set_title("E7 Family B — Domain-Shift / OOD Effects")
ax.grid(True, alpha=0.25)
fig.tight_layout()

fig.savefig(
    OUT_DIR / "Figure_E7_FamilyB_DomainShift_Effects.png",
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    OUT_DIR / "Figure_E7_FamilyB_DomainShift_Effects.pdf",
    bbox_inches="tight",
)
plt.show()


## 16. Automatic audit summary

This summary reports what the statistics support without changing the predeclared test families.


In [ ]:
def direction_text(delta, metric, effect_definition):
    if metric in ["brier", "nll"]:
        if effect_definition.startswith("intervention"):
            if delta < 0:
                return "lower proper loss under intervention"
            if delta > 0:
                return "higher proper loss under intervention"
            return "no proper-loss difference"
        if effect_definition == "ood_minus_in_domain":
            if delta > 0:
                return "higher proper loss under OOD"
            if delta < 0:
                return "lower proper loss under OOD"
            return "no proper-loss difference"
    return ""

summary_lines = [
    "# E7 Cluster-Aware Statistical Inference Summary",
    "",
    f"- Official-test rows: {EXPECTED_N}",
    f"- Unique post_id clusters: {N_CLUSTERS}",
    f"- Cluster bootstrap replicates: {N_BOOT}",
    f"- Cluster sign-flip draws: {N_SIGNFLIP}",
    "",
    "## Family A — cross-backbone proper scores",
]

for _, r in family_a.iterrows():
    summary_lines.append(
        f"- {r['backbone']} | {r['comparison']} | {r['metric']}: "
        f"Δ={r['delta']:.6f}, 95% CI [{r['ci95_low']:.6f}, {r['ci95_high']:.6f}], "
        f"Holm p={r['holm_p']:.6g}, "
        f"{direction_text(r['delta'], r['metric'], r['effect_definition'])}."
    )

summary_lines += [
    "",
    "## Family B — domain shift / OOD proper scores",
]

for _, r in family_b.iterrows():
    summary_lines.append(
        f"- {r['comparison']} | {r['metric']}: "
        f"Δ={r['delta']:.6f}, 95% CI [{r['ci95_low']:.6f}, {r['ci95_high']:.6f}], "
        f"Holm p={r['holm_p']:.6g}, "
        f"{direction_text(r['delta'], r['metric'], r['effect_definition'])}."
    )

summary_lines += [
    "",
    "## Scope limitation",
    "- Confidence intervals and p-values quantify clustered test-sample uncertainty conditional on the frozen fitted models.",
    "- They do not quantify retraining-seed / optimization uncertainty.",
    "- E6 fine-grained topic identities remain diagnostic because topic resolution was seed-sensitive; no individual topic hotspot is promoted by post-hoc p-values.",
]

(OUT_DIR / "E7_RESULTS_SUMMARY.md").write_text(
    "\n".join(summary_lines)
)

print("\n".join(summary_lines[:20]))
print("\n... summary saved to E7_RESULTS_SUMMARY.md")


## 17. Package hashes and final E7 archive

In [ ]:
def hash_output_files(directory):
    rows = []

    for p in sorted(Path(directory).iterdir()):
        if not p.is_file():
            continue

        if p.name in {
            "E7_OUTPUT_HASHES.csv",
            "E7_MANIFEST.json",
        }:
            continue

        rows.append({
            "filename": p.name,
            "bytes": p.stat().st_size,
            "sha256": sha256_file(p),
        })

    return pd.DataFrame(rows)

hashes = hash_output_files(OUT_DIR)

hashes.to_csv(
    OUT_DIR / "E7_OUTPUT_HASHES.csv",
    index=False,
)

manifest = {
    "package": "CalibBERT_v2_E7_Cluster_Aware_Inference",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "input_archives": e7_lock["inputs"],
    "n_test_rows": EXPECTED_N,
    "n_post_id_clusters": N_CLUSTERS,
    "bootstrap_replicates": N_BOOT,
    "signflip_draws": N_SIGNFLIP,
    "files": hashes.to_dict(orient="records"),
}

(OUT_DIR / "E7_MANIFEST.json").write_text(
    json.dumps(manifest, indent=2)
)

ZIP_PATH = PROJECT_ROOT / "E7_RESULTS.zip"

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as z:
    for p in sorted(OUT_DIR.iterdir()):
        if p.is_file():
            z.write(p, arcname=p.name)

print("Final E7 archive:", ZIP_PATH)
print("Output files:", len(list(OUT_DIR.iterdir())))


## 18. E7 completion gate

E7 is ready for final audit when:

- [ ] final frozen E3/E4/E6 archives are used
- [ ] all 715 official-test rows align exactly across E3 and E4
- [ ] `post_id` is used as the cluster unit
- [ ] E7 analysis lock is written before inferential results
- [ ] 10,000 cluster-bootstrap replicates complete
- [ ] 10,000 paired cluster sign-flip draws complete
- [ ] Family A contains exactly 12 proper-score tests
- [ ] Family B contains exactly 10 proper-score tests
- [ ] Holm correction is applied separately within Family A and Family B
- [ ] ECE is not treated as a primary inferential endpoint
- [ ] F1/AURC are reported with CIs as secondary endpoints
- [ ] LODO selective inference uses macro-domain AURC rather than pooled cross-fold ranking
- [ ] E6 topic evidence remains stability/diagnostic rather than post-hoc confirmatory
- [ ] model-training randomness is explicitly outside the E7 uncertainty scope
- [ ] `E7_RESULTS.zip` is safely retained in Google Drive

### Files to send for final E7 audit

1. `E7_RESULTS.zip`
2. `Table_E7_FamilyA_CrossBackbone_ProperScores.csv`
3. `Table_E7_FamilyB_DomainShift_ProperScores.csv`
4. `Table_E7_Secondary_E3_F1_AURC.csv`
5. `Table_E7_Secondary_E4_MacroDomain_AURC.csv`
6. `Table_E7_Integrated_Evidence_E2_to_E6.csv`
7. `E7_ANALYSIS_LOCK.json`

After E7 passes, the empirical reconstruction phase is complete.  
The next step is **E8 — integrated evidence lock, final claim matrix, and manuscript reconstruction**.
